In [1]:
# -- 0. SETUP --
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (StringType, IntegerType, FloatType, ArrayType,
                                StructType, StructField, LongType, DoubleType)

spark = SparkSession.builder \
    .appName("MyDigitalTwin-BehavioralClustering") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

import sys as _sys, os as _os
_sys.path.insert(0, _os.path.abspath(_os.path.join(_os.path.dirname("__file__"), "../../..")))
from config import WAREHOUSE

print(f"Warehouse: {WAREHOUSE}")
assert os.path.exists(WAREHOUSE), f"Warehouse introuvable: {WAREHOUSE}"

def read_table(name):
    return spark.read.format("delta").load(os.path.join(WAREHOUSE, name))

Warehouse: /opt/spark/data/warehouse


26/04/19 00:15:56 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


---
## PARTIE B — Behavioral Clustering

On rassemble toutes les activités avec leurs features temporelles : heure, jour de la semaine, plateforme, poids d'interaction.

**Features retenues (V1 finale)** : `hour_sin/cos`, `weekday_sin/cos`, `weight`, `platform_ohe`

> **Note V2 testée** : retrait de la plateforme pour des profils purement temporels → 6 clusters quasi-identiques (doublons "Soir Semaine" × 2, "Après-midi Weekend" × 2, Silhouette 0.33). La plateforme est un signal comportemental réel — mode Spotify journée ≠ soirée multi-plateforme. V1 conservée.

In [2]:
# ── B1. CHARGEMENT DES FEATURES COMPORTEMENTALES ──────────────────────────────
# Features retenues : hour (cyclique), weekday (cyclique), platform (OHE), weight
# Sources avec colonnes event_hour + event_weekday disponibles
#
# ⚠ Après ajout de nouvelles sources, relancer B3-B5 et mettre à jour BEH_LABELS
#   en lisant les résultats de B4 (plateforme dominante + heure par cluster).

def safe_read(name, hour_col="event_hour", weekday_col="event_weekday",
              platform_name=None, weight_val=None, limit=None):
    """Charge une table warehouse et retourne (hour, weekday, platform, weight)."""
    try:
        df = read_table(name)
        w_col = F.col("interaction_weight").cast(FloatType()) if "interaction_weight" in df.columns \
                else F.lit(weight_val or 1.0).cast(FloatType())
        sel = df.select(
            F.col(hour_col).alias("hour"),
            F.col(weekday_col).alias("weekday"),
            F.lit(platform_name or name).alias("platform"),
            w_col.alias("weight"),
        )
        if limit:
            sel = sel.limit(limit)
        return sel
    except Exception as e:
        print(f"⚠ {name} ignoré : {e}")
        return None

sources = [
    # ── Google / YouTube ──────────────────────────────────────────────────────
    safe_read("youtube_watch",    platform_name="youtube"),
    safe_read("google_searches",  platform_name="google",   weight_val=1.0),
    safe_read("google_chrome",    platform_name="chrome",   weight_val=1.0),
    # ── Spotify ───────────────────────────────────────────────────────────────
    safe_read("spotify_streams",  hour_col="listen_hour", weekday_col="listen_weekday",
              platform_name="spotify"),
    # ── Netflix (pas d'heure → 21h par défaut) ───────────────────────────────
    read_table("netflix_views").select(
        F.lit(21).cast(IntegerType()).alias("hour"),
        F.col("watch_weekday").alias("weekday"),
        F.lit("netflix").alias("platform"),
        F.col("interaction_weight").cast(FloatType()).alias("weight"),
    ),
    # ── TikTok — 3 sources (limites pour éviter de noyer) ────────────────────
    safe_read("tiktok_watch",     platform_name="tiktok",        limit=2000),
    safe_read("tiktok_likes",     platform_name="tiktok_likes",  limit=2000),
    safe_read("tiktok_searches",  platform_name="tiktok_search", weight_val=1.0, limit=1000),
    # ── Instagram — 4 sources (limites) ──────────────────────────────────────
    safe_read("instagram_likes",    platform_name="instagram",        limit=2000),
    safe_read("instagram_saved",    platform_name="instagram_saved",  limit=500),
    safe_read("instagram_comments", platform_name="instagram_comment",
              weight_val=2.5, limit=500),
    # Nouvelles tables (disponibles après re-ingestion instagram.ipynb)
    safe_read("instagram_posts_viewed",   platform_name="ig_posts",   limit=2000),
    safe_read("instagram_videos_watched", platform_name="ig_videos",  limit=2000),
    safe_read("instagram_story_likes",    platform_name="ig_stories",
              weight_val=1.5, limit=1000),
    safe_read("instagram_searches",       platform_name="ig_search",
              weight_val=1.0, limit=500),
    # ── Twitter ───────────────────────────────────────────────────────────────
    safe_read("twitter_tweets",   platform_name="twitter"),
]

valid_sources = [s for s in sources if s is not None]
behavioral_raw = valid_sources[0]
for s in valid_sources[1:]:
    behavioral_raw = behavioral_raw.union(s)

behavioral_raw = behavioral_raw.filter(
    F.col("hour").isNotNull() & F.col("weekday").isNotNull()
)

print(f"Total events comportementaux : {behavioral_raw.count():,}")
behavioral_raw.groupBy("platform").count().orderBy(F.desc("count")).show(20)


26/04/19 00:16:11 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Total events comportementaux : 117,814


+-----------------+-----+
|         platform|count|
+-----------------+-----+
|           google|55854|
|          spotify|33972|
|          youtube|13821|
|          netflix| 4288|
|           tiktok| 2000|
|     tiktok_likes| 2000|
|        instagram| 2000|
|    tiktok_search| 1000|
|        ig_videos|  994|
|         ig_posts|  820|
|       ig_stories|  366|
|           chrome|  338|
|          twitter|  319|
|instagram_comment|   28|
|  instagram_saved|   13|
|        ig_search|    1|
+-----------------+-----+



In [3]:
# -- B2. PROFILS TEMPORELS PAR PLATEFORME --
# Chaque plateforme devient un vecteur de 5 features :
# (% matin, % apres-midi, % soir, % nuit, % weekend)
# K-Means sur 16 profils au lieu de 226k events bruts.

from pyspark.ml.feature import VectorAssembler, StandardScaler

profiled = behavioral_raw \
    .withColumn("slot",
        F.when((F.col("hour") >= 5)  & (F.col("hour") < 12), "morning")
         .when((F.col("hour") >= 12) & (F.col("hour") < 18), "afternoon")
         .when((F.col("hour") >= 18) & (F.col("hour") < 23), "evening")
         .otherwise("night")
    ) \
    .withColumn("is_weekend", F.when(F.col("weekday") >= 6, 1.0).otherwise(0.0))

platform_total = profiled.groupBy("platform").agg(
    F.count("*").alias("total"),
    F.round(F.avg("hour"), 2).alias("avg_hour"),
    F.round(F.avg("weekday"), 2).alias("avg_weekday"),
    (F.sum("is_weekend") / F.count("*")).alias("weekend_pct"),
)

slot_counts = profiled.groupBy("platform", "slot").agg(F.count("*").alias("cnt"))
slot_pct = slot_counts.join(platform_total.select("platform", "total"), "platform") \
    .withColumn("pct", F.col("cnt") / F.col("total")) \
    .groupBy("platform") \
    .pivot("slot", ["morning", "afternoon", "evening", "night"]) \
    .agg(F.first("pct")) \
    .fillna(0.0)

platform_profiles = slot_pct.join(platform_total, "platform")

assembler = VectorAssembler(
    inputCols=["morning", "afternoon", "evening", "night", "weekend_pct"],
    outputCol="raw_features"
)
platform_profiles = assembler.transform(platform_profiles)

scaler = StandardScaler(inputCol="raw_features", outputCol="features", withMean=True, withStd=True)
scaler_model = scaler.fit(platform_profiles)
platform_profiles = scaler_model.transform(platform_profiles)

print(f"Profils de plateformes : {platform_profiles.count()} plateformes")
platform_profiles.select("platform", "morning", "afternoon", "evening", "night", "weekend_pct", "avg_hour").show(20, truncate=False)

Profils de plateformes : 16 plateformes
+-----------------+---------------------+-------------------+-------------------+-------------------+-------------------+--------+
|platform         |morning              |afternoon          |evening            |night              |weekend_pct        |avg_hour|
+-----------------+---------------------+-------------------+-------------------+-------------------+-------------------+--------+
|spotify          |0.23307429647945366  |0.32020487460261393|0.24269987048157307|0.20402095843635937|0.2900035323207347 |12.7    |
|tiktok_likes     |0.184                |0.272              |0.41               |0.134              |0.4005             |14.33   |
|ig_videos        |0.19416498993963782  |0.3782696177062374 |0.3470824949698189 |0.08048289738430583|0.4969818913480885 |14.35   |
|ig_stories       |0.22404371584699453  |0.31693989071038253|0.3825136612021858 |0.07650273224043716|0.2896174863387978 |15.02   |
|ig_search        |0.0                  |0.

In [4]:
# -- B3. KMEANS SUR PROFILS DE PLATEFORMES (k=4) --
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

K_BEHAVIORAL = 4

kmeans = KMeans(featuresCol="features", predictionCol="beh_cluster",
                k=K_BEHAVIORAL, seed=42, maxIter=100)

print(f"Training K-Means sur profils (k={K_BEHAVIORAL})...")
km_model = kmeans.fit(platform_profiles)
beh_df = km_model.transform(platform_profiles)

evaluator = ClusteringEvaluator(featuresCol="features", predictionCol="beh_cluster")
sil = evaluator.evaluate(beh_df)
print(f"Silhouette Score (profils): {sil:.4f}")

beh_df.select("platform", "beh_cluster", "avg_hour", "weekend_pct",
              "morning", "afternoon", "evening", "night") \
    .orderBy("beh_cluster", "platform").show(20, truncate=False)

Training K-Means sur profils (k=4)...


26/04/19 00:17:58 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


Silhouette Score (profils): 0.4807
+-----------------+-----------+--------+-------------------+---------------------+-------------------+-------------------+-------------------+
|platform         |beh_cluster|avg_hour|weekend_pct        |morning              |afternoon          |evening            |night              |
+-----------------+-----------+--------+-------------------+---------------------+-------------------+-------------------+-------------------+
|chrome           |0          |19.54   |0.05917159763313609|0.0029585798816568047|0.15680473372781065|0.5355029585798816 |0.3047337278106509 |
|google           |0          |15.41   |0.28644322698463853|0.18167006839259497  |0.36022487198768216|0.362964156551008  |0.09514090306871487|
|ig_posts         |0          |14.31   |0.4926829268292683 |0.2146341463414634   |0.3182926829268293 |0.3121951219512195 |0.1548780487804878 |
|ig_stories       |0          |15.02   |0.2896174863387978 |0.22404371584699453  |0.31693989071038253|0.382

In [5]:
# -- B4. CARACTERISATION DES CLUSTERS --

beh_cluster_info = []
cluster_data = beh_df.groupBy("beh_cluster").agg(
    F.collect_list("platform").alias("platforms"),
    F.round(F.avg("avg_hour"), 1).alias("avg_hour"),
    F.round(F.avg("avg_weekday"), 1).alias("avg_weekday"),
    F.round(F.avg("weekend_pct"), 3).alias("weekend_pct"),
    F.sum("total").alias("item_count"),
).orderBy("beh_cluster").collect()

for row in cluster_data:
    cid    = row["beh_cluster"]
    avg_h  = row["avg_hour"] or 0.0
    avg_wd = row["avg_weekday"] or 0.0
    count  = row["item_count"]
    plats  = row["platforms"]

    h = round(avg_h)
    if 5 <= h < 12:    period = "Matin"
    elif 12 <= h < 18: period = "Apres-midi"
    elif 18 <= h < 23: period = "Soir"
    else:              period = "Nuit"

    day_type = "Weekend" if row["weekend_pct"] > 0.35 else "Semaine"

    beh_cluster_info.append({
        "cluster_id":    cid,
        "item_count":    count,
        "avg_hour":      float(avg_h),
        "avg_weekday":   float(avg_wd),
        "time_period":   period,
        "day_type":      day_type,
        "top_platforms": plats,
    })

    print(f"[Cluster {cid}] {count:,} events | {period} . {day_type} | {plats}")

[Cluster 0] 111,512 events | Apres-midi . Semaine | ['youtube', 'google', 'chrome', 'spotify', 'tiktok_likes', 'tiktok_search', 'instagram', 'instagram_comment', 'ig_posts', 'ig_videos', 'ig_stories', 'twitter']
[Cluster 1] 4,288 events | Soir . Semaine | ['netflix']
[Cluster 2] 1 events | Nuit . Weekend | ['ig_search']
[Cluster 3] 2,013 events | Apres-midi . Semaine | ['tiktok', 'instagram_saved']


In [6]:
# -- B5. LABELS DES CLUSTERS --
# Placeholders — a ajuster apres lecture des resultats B4.

BEH_LABELS = {
      0: {"label": "🌐 Navigation & Social · Journée",  "emoji": "🌐"},
      1: {"label": "🎬 Netflix · Soirée",                "emoji": "🎬"},
      2: {"label": "👻 Activité rare",                   "emoji": "👻"},
      3: {"label": "📱 TikTok · Après-midi",             "emoji": "📱"},
}

for info in beh_cluster_info:
    cid = info["cluster_id"]
    print(f"  Cluster {cid}: {info['time_period']} . {info['day_type']} | {info['top_platforms']}")

  Cluster 0: Apres-midi . Semaine | ['youtube', 'google', 'chrome', 'spotify', 'tiktok_likes', 'tiktok_search', 'instagram', 'instagram_comment', 'ig_posts', 'ig_videos', 'ig_stories', 'twitter']
  Cluster 1: Soir . Semaine | ['netflix']
  Cluster 2: Nuit . Weekend | ['ig_search']
  Cluster 3: Apres-midi . Semaine | ['tiktok', 'instagram_saved']


In [7]:
# -- B6. ECRITURE behavioral_clusters (Delta) --

beh_rows = []
for info in beh_cluster_info:
    cid = info["cluster_id"]
    beh_rows.append((
        cid,
        BEH_LABELS.get(cid, {}).get("label", f"Profil {cid}"),
        BEH_LABELS.get(cid, {}).get("emoji", "?"),
        float(info["avg_hour"]),
        float(info["avg_weekday"]),
        info["time_period"],
        info["day_type"],
        info["top_platforms"],
        info["item_count"]
    ))

schema_beh = StructType([
    StructField("cluster_id",    IntegerType(), False),
    StructField("label",         StringType(),  False),
    StructField("emoji",         StringType(),  True),
    StructField("avg_hour",      DoubleType(),  True),
    StructField("avg_weekday",   DoubleType(),  True),
    StructField("time_period",   StringType(),  True),
    StructField("day_type",      StringType(),  True),
    StructField("top_platforms", ArrayType(StringType()), True),
    StructField("item_count",    LongType(),    True),
])

beh_clusters_df = spark.createDataFrame(beh_rows, schema_beh)
out_path = os.path.join(WAREHOUSE, "behavioral_clusters")
beh_clusters_df.write.format("delta").mode("overwrite").save(out_path)

print(f"Ecrit dans : {out_path}")
beh_clusters_df.show(truncate=50)

Ecrit dans : /opt/spark/data/warehouse/behavioral_clusters
+----------+--------------------------------+-----+--------+-----------+-----------+--------+--------------------------------------------------+----------+
|cluster_id|                           label|emoji|avg_hour|avg_weekday|time_period|day_type|                                     top_platforms|item_count|
+----------+--------------------------------+-----+--------+-----------+-----------+--------+--------------------------------------------------+----------+
|         0|🌐 Navigation & Social · Journée|   🌐|    15.3|        4.0| Apres-midi| Semaine|[youtube, google, chrome, spotify, tiktok_likes...|    111512|
|         1|             🎬 Netflix · Soirée|   🎬|    21.0|        4.0|       Soir| Semaine|                                         [netflix]|      4288|
|         2|                👻 Activité rare|   👻|     0.0|        6.0|       Nuit| Weekend|                                       [ig_search]|         1|
|         3

In [8]:
spark.stop()
print("Spark session fermée. Notebook terminé.")

Spark session fermée. Notebook terminé.
